In [30]:
pip install sportsdataverse

Note: you may need to restart the kernel to use updated packages.


In [31]:
pip install polars

Note: you may need to restart the kernel to use updated packages.


In [32]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [33]:
import polars as pl
import numpy as np
from scipy.optimize import minimize_scalar
from scipy.special import expit

import requests


In [34]:
def get_nl_schedule(season=2027, start_date="15.09.2026", end_date="01.03.2027", league=1, region="all"):
    url = (
        "https://data.sihf.ch/Statistic/api/cms/cache300"
        f"?alias=results&searchQuery=1,10/2015-2099/1,2,3,4,5,14,15,16,10,18,19,23,24,25,26,28,27,29,30,31,32,33,35,36,27,28,29,40,41,42,43,44,45,46,60,61,83,100,101,104105,106,107,113,114,115,116,117,118,119,120,121,122,123,124,125"
        f"&filterQuery={season}/{league}/{region}/{start_date}-{end_date}"
        "&filterBy=season,league,region,date&language=fr&format=json"
    )
    data = requests.get(url).json()["data"]

    rows = []
    for _, date, _, home, away, result, _, _, _, link, _ in data:
        rows.append({
            "date": date,
            "home_id": home["id"],
            "home_name": home["name"],
            "home_acronym": home["acronym"],
            "away_id": away["id"],
            "away_name": away["name"],
            "away_acronym": away["acronym"],
            "home_score": result["homeTeam"],
            "away_score": result["awayTeam"],
            "game_id": link["gameId"],
        })

    df = pl.DataFrame(rows).with_columns(
        pl.col("date").str.strptime(pl.Date, "%d.%m.%Y"),
        pl.col("home_score", "away_score").cast(pl.Int32, strict=False),
    )
    return df

In [35]:
sched = get_nl_schedule().drop_nulls(subset=["home_score", "away_score"])
sched 

date,home_id,home_name,home_acronym,away_id,away_name,away_acronym,home_score,away_score,game_id
date,i64,str,str,i64,str,str,i32,i32,str
2026-09-15,101150,"""HC Lugano""","""HCL""",103140,"""Genève-Servette HC""","""GSHC""",6,3,"""20271105000006"""
2026-09-15,101060,"""SC Rapperswil-Jona Lakers""","""SCRJ""",101149,"""EHC Kloten""","""EHCK""",3,4,"""20271105000004"""
2026-09-15,101151,"""HC Davos""","""HCD""",102127,"""SCL Tigers""","""SCL""",4,1,"""20271105000003"""
2026-09-15,102128,"""EHC Biel-Bienne""","""EHCB""",103138,"""Fribourg-Gottéron""","""FRI""",4,2,"""20271105000002"""
2026-09-15,103141,"""Lausanne HC""","""LHC""",101144,"""EV Zug""","""EVZ""",0,5,"""20271105000005"""
…,…,…,…,…,…,…,…,…,…
2026-10-02,103141,"""Lausanne HC""","""LHC""",102128,"""EHC Biel-Bienne""","""EHCB""",7,4,"""20271105000054"""
2026-10-02,101151,"""HC Davos""","""HCD""",103144,"""HC Ajoie""","""HCA""",3,1,"""20271105000052"""
2026-10-02,102126,"""SC Bern""","""SCB""",103140,"""Genève-Servette HC""","""GSHC""",2,3,"""20271105000051"""


## Let's build matrice X and vector y

In [36]:
import numpy as np
from numpy.linalg import inv

In [37]:
y = sched.select(["home_score", "away_score"])
y.insert_column(2, (pl.col("home_score") - pl.col("away_score")).alias("score_diff"))
y = y.select(["score_diff"])
y = np.matrix(y.to_numpy())
print(y, y.shape)

[[ 3]
 [-1]
 [ 3]
 [ 2]
 [-5]
 [ 2]
 [-1]
 [ 2]
 [-1]
 [ 3]
 [ 3]
 [ 3]
 [-1]
 [-4]
 [-1]
 [ 1]
 [ 3]
 [ 1]
 [ 3]
 [-2]
 [-2]
 [-1]
 [-3]
 [ 1]
 [ 2]
 [-1]
 [-6]
 [ 1]
 [ 2]
 [-1]
 [ 1]
 [-3]
 [ 1]
 [ 2]
 [ 1]
 [-1]
 [-2]
 [-2]
 [ 4]
 [-3]
 [ 3]
 [-3]
 [ 3]
 [ 1]
 [ 3]
 [ 1]
 [-4]
 [ 3]
 [ 2]
 [-1]
 [-3]
 [ 3]] (52, 1)


In [38]:
teams = sorted(set(sched["home_acronym"]) | set(sched["away_acronym"]))
team_idx = {t: j + 1 for j, t in enumerate(teams)}   # column 0 is the intercept

n, p = sched.height, len(teams) + 1
X = np.zeros((n, p))
X[:, 0] = 1                                            # intercept

rows = np.arange(n)
X[rows, sched["home_acronym"].replace_strict(team_idx).to_numpy()] = 1
X[rows, sched["away_acronym"].replace_strict(team_idx).to_numpy()] = -1

X = np.matrix(X)
print(X, X.shape)

[[ 1.  0.  0.  0.  0. -1.  0.  0.  0.  1.  0.  0.  0.  0.  0.]
 [ 1.  0. -1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  1.  0.]
 [ 1.  0.  0.  0.  0.  0.  0.  0.  1.  0.  0.  0. -1.  0.  0.]
 [ 1.  1.  0.  0. -1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 1.  0.  0. -1.  0.  0.  0.  0.  0.  0.  1.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. -1.  0.  0.  1.]
 [ 1.  0.  0.  0.  0.  0.  1. -1.  0.  0.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  1.  0.  0.  0.  0.  0. -1.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  0.  1. -1.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  0.  0.  0.  1.  0.  0.  0.  0.  0. -1.  0.]
 [ 1.  0.  1.  0.  0.  0.  0.  0.  0. -1.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  0.  0.  0.  0.  0. -1.  0.  0.  1.  0.  0.  0.]
 [ 1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  1.  0. -1.]
 [ 1.  0.  0. -1.  0.  0.  0.  0.  1.  0.  0.  0.  0.  0.  0.]
 [ 1.  1.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. -1.  0.  0.]
 [ 1.  0. -1.  0.  0.  0.  1.  0.  0.  0.  0.  0.  0.  

In [39]:
ref = 1 
X_red = np.delete(X, ref, axis=1)

B_red = np.linalg.solve(X_red.T @ X_red, X_red.T @ y)

B = np.insert(B_red, ref, 0.0)


In [42]:
B_flat = np.asarray(B).ravel()

home_adv = B_flat[0]
strengths = B_flat[1:] - B_flat[1:].mean()


strength_df = pl.DataFrame({
    "team_accronym": teams,
    "strength": strengths,
}).sort("strength", descending=True)

strength_df = strength_df.with_columns(
    pl.col("strength")
      .rank(method="ordinal", descending=True)
      .alias("strength_rank")
)

print(f"home advantage: {home_adv:.3f}")
with pl.Config(tbl_rows=-1):   
    print(strength_df)

home advantage: 0.418
shape: (14, 3)
┌───────────────┬───────────┬───────────────┐
│ team_accronym ┆ strength  ┆ strength_rank │
│ ---           ┆ ---       ┆ ---           │
│ str           ┆ f64       ┆ u32           │
╞═══════════════╪═══════════╪═══════════════╡
│ EVZ           ┆ 2.032089  ┆ 1             │
│ ZSC           ┆ 1.684706  ┆ 2             │
│ FRI           ┆ 0.487755  ┆ 3             │
│ SCL           ┆ 0.484778  ┆ 4             │
│ HCD           ┆ 0.072591  ┆ 5             │
│ GSHC          ┆ 0.057322  ┆ 6             │
│ SCB           ┆ -0.135377 ┆ 7             │
│ HCL           ┆ -0.153224 ┆ 8             │
│ LHC           ┆ -0.164838 ┆ 9             │
│ EHCB          ┆ -0.336458 ┆ 10            │
│ HCA           ┆ -0.372391 ┆ 11            │
│ HCAP          ┆ -0.744394 ┆ 12            │
│ SCRJ          ┆ -1.170312 ┆ 13            │
│ EHCK          ┆ -1.742247 ┆ 14            │
└───────────────┴───────────┴───────────────┘


Zug Being at the top of the strength is a suprise because they currently are 9th in the league standing, it maybe the small samples that make it so. They won against Lausanne(8th in standing, 9th strength) Davos(4th in standing, 5th strenght) and Langnau(5th standing, 4th strength) and lost against Lugano(11th standing, 8th strength) and Langnau. Zug are the only team with 5 GP all other teams have played 6+ games with only 2 team having 6GP.

The other surprise is Ajoie having the 11th strength while beeing 7th in standing with 8GP. 

Also Geneve is 2th in standing but 6th in strength indicates that their schedule for their 9 first game was easier than the rest of the league.

In [44]:
standing = {"ZSC":1, "GSHC":2, "FRI":3,"HCD":4,"SCL":5, "HCAP":6,"HCA":7, "LHC":8,"EVZ":9,"SCRJ":10, "HCL":11,"SCB":12,"EHCB":12,"EHCK":14}
standing

{'ZSC': 1,
 'GSHC': 2,
 'FRI': 3,
 'HCD': 4,
 'SCL': 5,
 'HCAP': 6,
 'HCA': 7,
 'LHC': 8,
 'EVZ': 9,
 'SCRJ': 10,
 'HCL': 11,
 'SCB': 12,
 'EHCB': 12,
 'EHCK': 14}

In [45]:
strength_df = strength_df.with_columns(
    pl.col("team_accronym")
      .replace_strict(standing, return_dtype=pl.UInt32)
      .alias("standing")
)

with pl.Config(tbl_rows=-1):
    print(strength_df)

shape: (14, 4)
┌───────────────┬───────────┬───────────────┬──────────┐
│ team_accronym ┆ strength  ┆ strength_rank ┆ standing │
│ ---           ┆ ---       ┆ ---           ┆ ---      │
│ str           ┆ f64       ┆ u32           ┆ u32      │
╞═══════════════╪═══════════╪═══════════════╪══════════╡
│ EVZ           ┆ 2.032089  ┆ 1             ┆ 9        │
│ ZSC           ┆ 1.684706  ┆ 2             ┆ 1        │
│ FRI           ┆ 0.487755  ┆ 3             ┆ 3        │
│ SCL           ┆ 0.484778  ┆ 4             ┆ 5        │
│ HCD           ┆ 0.072591  ┆ 5             ┆ 4        │
│ GSHC          ┆ 0.057322  ┆ 6             ┆ 2        │
│ SCB           ┆ -0.135377 ┆ 7             ┆ 12       │
│ HCL           ┆ -0.153224 ┆ 8             ┆ 11       │
│ LHC           ┆ -0.164838 ┆ 9             ┆ 8        │
│ EHCB          ┆ -0.336458 ┆ 10            ┆ 12       │
│ HCA           ┆ -0.372391 ┆ 11            ┆ 7        │
│ HCAP          ┆ -0.744394 ┆ 12            ┆ 6        │
│ SCRJ          